# ac data

In [3]:
import pandas as pd
import re
import os
from striprtf.striprtf import rtf_to_text
from datetime import datetime

def read_rtf(file_path):
    with open(file_path, 'r') as file:
        rtf_content = file.read()
    text = rtf_to_text(rtf_content)

    return text

def get_year(text):
    year_pattern = r'\b(19\d{2}|20\d{2}|25\d{2})\b'
    years = re.findall(year_pattern, text)

    # find year lower than current year
    current_year = datetime.now().year
    years = [year for year in years if int(year) <= current_year]

    return years[0]

def clean_text(text):
    lines = text.split('\n')
    data_lines = [line for line in lines if line.strip()]
    data = []
    for line in data_lines:
        parsed_line = [x.strip() for x in line.split('\t')]
        # if first element of line is empty, remove it
        if not parsed_line[0]:
            parsed_line = parsed_line[1:]
        # if first element of line is Zone, combine it with next element
        if parsed_line[0] == 'Zone':
            parsed_line[1] = 'Zone:' + parsed_line[1]
            parsed_line = parsed_line[1:]
        #
        if parsed_line[0] == 'Zone:':
            parsed_line[1] = 'Zone:' + parsed_line[1]
            parsed_line = parsed_line[1:]
        # if first element of line is Age, combine it with next element
        if parsed_line:
            data.append(parsed_line)

    # get year
    year = get_year(text)

    # remove first and last line
    data = data[1:-1]
    return data, year

def fill_name(col_names):
    cleaned_names = []

    for i, name in enumerate(col_names):
        name = name.strip()
        if 'Area' in name:
            cleaned_names.append('Areas')
        elif 'Total' in name:
            cleaned_names.append('Total')
        elif '<1' in name or '<1ปี' in name:
            cleaned_names.append('<1')
        elif '65+' in name:
            cleaned_names.append('65+')
        elif 'ไม่ทรา' in name:
            cleaned_names.append('Unknown')
        elif '+' in name:
            current = name.strip('+').strip()
            if i + 1 < len(col_names) and '+' in col_names[i + 1] and not col_names[i + 1].endswith('+'):
                try:
                    next_age = int(re.sub(r'\D', '', col_names[i + 1]))
                    cleaned_names.append(f"{current}-{next_age - 1}")
                except ValueError:
                    cleaned_names.append(f"{current}-{current}")  # if conversion fails, use current as is
            else:
                cleaned_names.append(f"{current}-{current}")  # No next range, so make it a self range
        elif name.endswith('-'):
            current = name.strip('-').strip()
            if i + 1 < len(col_names):
                # Extract the first sequence of digits from the next column's name
                next_age_match = re.search(r'\d+', col_names[i + 1])
                if next_age_match:
                    next_age = int(next_age_match.group())
                    cleaned_names.append(f"{current}-{next_age - 1}")
                else:
                    cleaned_names.append(current)  # Use current if no digits found
            else:
                cleaned_names.append(current)  # Use current as is if no next info
        else:
            cleaned_names.append(name)  # Directly use clear ranges like '10-14'
    
    total_index = cleaned_names.index('Total')
    under_one_index = cleaned_names.index('<1')
    if under_one_index - total_index == 2:
        cleaned_names[total_index + 1] = '0'

    return cleaned_names

def likely_table(row):
    numeric_count = sum(x.replace('.', '', 1).isdigit() for x in row if x)
    
    return numeric_count < len(row) / 2
    
def text_to_dataframe(data, year, disease):
    
    data = [[re.sub('ปี', '', x) for x in row] for row in data]

    # convert to dataframe
    data = pd.DataFrame(data[1:], columns=data[0])
    data = data.loc[:, data.apply(lambda col: not all(x == "" for x in col))]
    data.columns = fill_name(data.columns)

    # remove rows which likely contain header
    data = data[~data.apply(likely_table, axis=1)]

    # convert to longer format
    data = data.melt(id_vars=['Areas'], var_name='Age', value_name='Cases')
    data['Year'] = year
    data['Disease'] = disease

    # remove spaces in Cases column and convert to integer
    data['Cases'] = data['Cases'].str.replace(',', '')
    data['Cases'] = data['Cases'].astype(int)

    return data

In [5]:
# TEST
file_path = '../Data/GetData/Rubella/ac_Rubella_50.rtf'
text = read_rtf(file_path)
data, year = clean_text(text)
df = text_to_dataframe(data, year, 'Rubella')
df

,Areas,Age,Cases,Year,Disease
0,Total,Total,341,2007,Rubella
1,North Region,Total,36,2007,Rubella
2,Zone: 1,Total,12,2007,Rubella
3,Chiang Mai,Total,6,2007,Rubella
4,Chiang Rai,Total,2,2007,Rubella
...,...,...,...,...,...
1795,Pattani,Unknown,0,2007,Rubella
1796,Yala,Unknown,0,2007,Rubella
1797,Zone: 19,Unknown,0,2007,Rubella
1798,Satun,Unknown,0,2007,Rubella


In [3]:
# get disease list
disease_list = os.listdir('../Data/GetData/')
for disease in disease_list:
    file_list = os.listdir('../Data/GetData/' + disease)
    file_list = [file for file in file_list if file.startswith('ac_')]

    if not file_list:
        continue
    
    outcome = []
    try:
        for file in file_list:
            file_path = f'../Data/GetData/{disease}/{file}'
            text = read_rtf(file_path)
            data, year = clean_text(text)
            df = text_to_dataframe(data, year, disease)
            outcome.append(df)

        outcome = pd.concat(outcome)
        outcome.to_csv(f'../Data/CleanData/{disease}_ac.csv', index=False)
    except Exception as e:
        print(f'* Error in {disease}-{file_path}: {e}')
        continue

# ad data

In [4]:
# TEST
file_path = '../Data/GetData/Zika virus/ad_Zika_67.rtf'
text = read_rtf(file_path)
data, year = clean_text(text)
df = text_to_dataframe(data, year, 'Zika virus')

In [5]:
disease_list = os.listdir('../Data/GetData/')
for disease in disease_list:
    file_list = os.listdir('../Data/GetData/' + disease)
    file_list = [file for file in file_list if file.startswith('ad_')]

    if not file_list:
        continue

    outcome = []
    try:
        for file in file_list:
            file_path = f'../Data/GetData/{disease}/{file}'
            text = read_rtf(file_path)
            data, year = clean_text(text)
            df = text_to_dataframe(data, year, disease)
            outcome.append(df)

        outcome = pd.concat(outcome)
        outcome.to_csv(f'../Data/CleanData/{disease}_ad.csv', index=False)
    except Exception as e:
        print(f'* Error in {disease}-{file}: {e}')
        continue

# mcd data

In [19]:
def text_to_dataframe(data, year, disease):
    # create headers
    headers = data[0]
    labels = data[1]
    adjusted_headers = ['Areas']
    for i, header in enumerate(headers[1:]):
        adjusted_headers.append(f'{header}_{labels[i*2-2]}')
        adjusted_headers.append(f'{header}_{labels[i*2-1]}')

    # convert to dataframe
    df = pd.DataFrame(data[2:], columns=adjusted_headers)
    df = df.loc[:, df.apply(lambda col: not all(x == "" for x in col))]
    
    # Melt the DataFrame to longer format
    df = df.melt(id_vars=['Areas'], var_name='Period_Type', value_name='Count')
    df[['Month', 'Type']] = df['Period_Type'].str.split('_', expand=True)
    df.drop('Period_Type', axis=1, inplace=True)

    # if Type contains cas, replace it with Cases
    df['Type'] = df['Type'].apply(lambda x: 'Cases' if 'cas' in x.lower() else x)
    df['Type'] = df['Type'].apply(lambda x: 'Deaths' if 'dea' in x.lower() else x)
    
    # Assign static values for 'Year' and 'Disease'
    df['Year'] = year
    df['Disease'] = disease

    # remove including NAN values
    df = df.dropna()

    # remove rows which not number in Count
    df = df[df['Count'].apply(lambda x: x.isnumeric())]
    df = df.astype({'Count': 'int32'})

    return df

In [20]:
# get disease list
disease_list = os.listdir('../Data/GetData/')
for disease in disease_list:
    file_list = os.listdir('../Data/GetData/' + disease)
    file_list = [file for file in file_list if file.startswith('mcd_')]

    if not file_list:
        continue

    outcome = []
    try:
        for file in file_list:
            file_path = f'../Data/GetData/{disease}/{file}'
            text = read_rtf(file_path)
            data, year = clean_text(text)
            df = text_to_dataframe(data, year, disease)
            outcome.append(df)

        outcome = pd.concat(outcome)
        outcome.to_csv(f'../Data/CleanData/{disease}_mcd.csv', index=False)
    except Exception as e:
        print(f'* Error in {disease}-{file}: {e}')
        continue

## rate

In [49]:
def fill_name(col_names):
    cleaned_names = []

    for i, name in enumerate(col_names):
        name = name.strip()
        if 'area' in name:
            cleaned_names.append('Areas')
        elif 'จำนวน' in name:
            cleaned_names.append('Population')
        elif 'cas' in name:
            cleaned_names.append('Cases')
        elif 'orbidity' in name:
            cleaned_names.append('Incidence')
        elif 'dea' in name:
            cleaned_names.append('Deaths')
        elif 'ortality' in name:
            cleaned_names.append('Mortality')
        elif 'CFR' in name:
            cleaned_names.append('CFR')
        elif 'dea' in name:
            cleaned_names.append('Deaths')
        else:
            cleaned_names.append(name)

    # select only columns with Areas, Cases, Incidence, Deaths, Mortality, CFR, Population
    cleaned_names = [name for name in cleaned_names if name in ['Areas', 'Cases', 'Incidence', 'Deaths', 'Mortality', 'CFR', 'Population']]

    return cleaned_names
    
def text_to_dataframe(data, year, disease):
    # create headers
    headers = data[0]
    headers = fill_name(headers)

    # drop rows which number of columns less than headers, except None values
    data = [row for row in data if len(row) == len(headers)]

    # convert to dataframe
    data = pd.DataFrame(data, columns=headers)
    data = data.loc[:, data.apply(lambda col: not all(x == "" for x in col))]

    # remove rows which likely contain header
    data = data[~data.apply(likely_table, axis=1)] 
    
    # Assign static values for 'Year' and 'Disease'
    data['Year'] = year
    data['Disease'] = disease

    return data

In [50]:
file = '../Data/GetData/Tetanus neonatorum/rate_TetanusNeo_47.rtf'
text = read_rtf(file)
data, year = clean_text(text)

# print frequency of columns
table = []
for row in data:
    table.append(len(row))

# frequency of each value
pd.Series(table).value_counts()

7     101
10      2
2       1
Name: count, dtype: int64

In [53]:
df = text_to_dataframe(data, year, 'disease')
df

,Areas,Cases,Incidence,Deaths,CFR,Mortality,Population,Year,Disease
0,Total,11,1.35,0,0.00,0.00,813069,2004,disease
1,North Region,3,2.52,0,0.00,0.00,119023,2004,disease
2,Zone:1,2,3.58,0,0.00,0.00,55793,2004,disease
3,Chiang Mai,1,5.22,0,0.00,0.00,19145,2004,disease
4,Chiang Rai,0,0.00,0,0.00,0.00,11608,2004,disease
...,...,...,...,...,...,...,...,...,...
96,Pattani,1,9.00,0,0.00,0.00,11110,2004,disease
97,Yala,0,0.00,0,0.00,0.00,10240,2004,disease
98,Zone:19,0,0.00,0,0.00,0.00,26020,2004,disease
99,Satun,0,0.00,0,0.00,0.00,4663,2004,disease


In [54]:
# get disease list
disease_list = os.listdir('../Data/GetData/')
for disease in disease_list:
    file_list = os.listdir('../Data/GetData/' + disease)
    file_list = [file for file in file_list if file.startswith('rate_')]

    if not file_list:
        continue

    outcome = []
    try:
        for file in file_list:
            file_path = f'../Data/GetData/{disease}/{file}'
            text = read_rtf(file_path)
            data, year = clean_text(text)
            df = text_to_dataframe(data, year, disease)
            outcome.append(df)

        outcome = pd.concat(outcome)
        outcome.to_csv(f'../Data/CleanData/{disease}_rate.csv', index=False)
    except Exception as e:
        print(f'* Error in {disease}-{file}: {e}')
        continue

* Error in Cholera-rate_Cholera_46.rtf: list index out of range
